In [ ]:
import cs
from glob import glob
import pandas as pd
from tqdm.notebook import tqdm
import json

In [ ]:
from sklearn.pipeline import Pipeline
from gsitk.preprocess import simple, Preprocessor, JoinTransformer
from gsitk.preprocess.stopwords import StopWordsRemover
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords

def preprocess(texts):
    preprocessing_pipe = Pipeline([
        ('preprocessor', Preprocessor(simple)),
        ('join', JoinTransformer()),
        ('stopwords', StopWordsRemover()),
    ])
    
    texts = preprocessing_pipe.fit_transform(texts)
    return texts

# Read data

In [ ]:
moral_names = ['base', 'care', 'equality', 'proportionality', 'loyalty', 'authority', 'purity']

generated_texts = list()
annotated_texts = list()

for moral_name in tqdm(moral_names):
    for filepath in glob("moral_boh/output/generations/**/*"):
        print(filepath)
        try:
            generated_texts.append((
                pd.read_json(filepath).loc[0,"generations"]["replies"][moral_name]["answer"],
                moral_name
            ))
        except KeyError:
            print("ERROR:", filepath)
    
    for filepath in glob("moral_boh/data/annotations/*.csv"):
        file_i = pd.read_csv(filepath)
        for filepath_i in file_i["label"].dropna().index.values:
            for anno in json.loads(file_i["label"].loc[filepath_i]):
                annotated_texts.append((anno["text"], anno["labels"][0]))

# Generate lexicon

## Preprocess

In [ ]:
generated_texts = pd.DataFrame(generated_texts, columns=["text", "label"])
annotated_texts = pd.DataFrame(annotated_texts, columns=["text", "label"])

# generated_texts["text_pp"] = 
texts_pp = list()
for text in list(generated_texts["text"].values):
    words = simple.preprocess(text)
    words = " ".join(words)
    texts_pp.append(words)
texts_pp = StopWordsRemover().fit_transform(texts_pp)
generated_texts["text_pp"] = texts_pp

texts_pp = list()
for text in list(annotated_texts["text"].values):
    words = simple.preprocess(text)
    words = " ".join(words)
    texts_pp.append(words)
texts_pp = StopWordsRemover().fit_transform(texts_pp)
annotated_texts["text_pp"] = texts_pp

## Generate

In [ ]:
generated_texts

In [ ]:
# generations
lexicon_generated, analyzer_generated = cs.generate_normalize_lexicon(
    annotations=pd.get_dummies(generated_texts['label']),
    texts=generated_texts['text_pp'],
    cutoff=10,
)

# annotations
lexicon_annotated, analyzer_annotated = cs.generate_normalize_lexicon(
    annotations=pd.get_dummies(annotated_texts['label']),
    texts=annotated_texts['text_pp'],
    cutoff=10,
)

## Export

In [ ]:
lexicon_generated["tfidf"].to_csv("lexicon_morality_from_generated-texts_v2.csv")
lexicon_annotated["tfidf"].to_csv("lexicon_morality_from_annotated-texts_v2.csv")